In [2]:
# ===============================================
# 🧠 Simple Groq LLM
# ===============================================
# This installs Groq’s Python SDK — required to connect to Groq LLMs
!pip install -q groq


In [8]:
from getpass import getpass
import os
from groq import Groq

# Securely input your API key (input is hidden)
os.environ["GROQ_API_KEY"] = getpass("Paste your GROQ API key (input hidden): ")

# Create a Groq client instance for model access
client = Groq(api_key=os.environ["GROQ_API_KEY"])

print("✅ Groq client ready (model: llama-3.3-70b-versatile)")


Paste your GROQ API key (input hidden): ··········
✅ Groq client ready (model: llama-3.3-70b-versatile)


In [4]:
# ===============================================
#  Dynamic text chunking
# ===============================================
# This function splits large text into overlapping chunks.
# You’ll use this later if you want to feed long documents.
# For now we keep the chatbot purely general-knowledge.

import re

def dynamic_chunk_text(text, target_tokens=300, overlap=50):
    """
    Dynamically split long text into overlapping word chunks.
    Args:
        text (str): full text to split
        target_tokens (int): approx. words per chunk
        overlap (int): number of overlapping words
    Returns:
        list[str]: chunks of text
    """
    if not text.strip():
        return []
    words = text.split()
    chunks, i = [], 0
    while i < len(words):
        chunk = words[i:i + target_tokens]
        chunks.append(" ".join(chunk))
        i += target_tokens - overlap
    return chunks

# Leave context blank → chatbot will rely on general knowledge only
extra_context = ""

# Prepare chunks if any context is ever added
context_chunks = dynamic_chunk_text(extra_context, target_tokens=120, overlap=20)
print(f"📄 Dynamic chunking ready. Context chunks loaded: {len(context_chunks)}")


📄 Dynamic chunking ready. Context chunks loaded: 0


In [5]:
# ===============================================
# – Define chat function using Groq LLM
# ===============================================
# This sends your prompt (and optional history/context) to the model.
# It supports short-term memory and optional custom context.

def chat_with_groq(prompt, history=None, extra_context=None,
                   model="llama-3.3-70b-versatile"):
    """
    Send user prompt to Groq LLM and return its reply.
      - history: optional [(role, content)] list for conversation memory
      - extra_context: optional additional context text/chunks
    """
    history = history or []
    messages = [{"role": "system", "content": "You are a helpful AI assistant."}]

    # Add extra context if provided
    if extra_context and extra_context.strip():
        context_text = "\n\n".join(extra_context) if isinstance(extra_context, list) else str(extra_context)
        messages.append({"role": "system", "content": f"Additional context:\n{context_text}"})

    # Include previous conversation for continuity
    for role, content in history:
        messages.append({"role": role, "content": content})

    # Current user prompt
    messages.append({"role": "user", "content": prompt})

    # Send to Groq API
    try:
        resp = client.chat.completions.create(
            model=model,
            messages=messages,
            max_tokens=400,
            temperature=0.6
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        return f"[Error contacting Groq API: {e}]"


In [7]:
# ===============================================
# 🤖 Step 5 – Start chatting!
# ===============================================
# Type any question to chat with the Groq model.
# It will answer from general knowledge (since context is blank).
# Type 'exit' or 'quit' to stop.

conversation_history = []  # remembers last few turns

print("\n🤖 Chatbot ready! Ask me anything, or type 'exit' to quit.\n")

while True:
    user_input = input("You: ").strip()
    if user_input.lower() in {"exit", "quit"}:
        print("Goodbye 👋")
        break

    # Get AI response
    response = chat_with_groq(user_input, history=conversation_history, extra_context=extra_context)
    print("AI:", response, "\n")

    # Save short conversation memory (last 10 messages)
    conversation_history.append(("user", user_input))
    conversation_history.append(("assistant", response))
    if len(conversation_history) > 10:
        conversation_history = conversation_history[-10:]



🤖 Chatbot ready! Ask me anything, or type 'exit' to quit.

You: exit
Goodbye 👋
